# Notebook - Phase 6

## Triage of the 11 problematic tickers

In [ ]:
import yaml
from pandas.core.frame import DataFrame

with open("/Users/hugo/quant-research/config/equity_universes.yaml", 'r') as f:
    config = yaml.safe_load(f)
universe = 'russell1000'

tickers = config[universe]
print(len(tickers))

In [ ]:
import pandas as pd
df = pd.read_csv("/Users/hugo/quant-research/config/russell1000_sectors.csv")
df.head()

In [ ]:
# check if symbols match
df_tolist = df["Symbol"].to_list()
print(set(df_tolist)-set(tickers))
print(set(tickers)-set(df_tolist))

In [ ]:
def normalize(x):
    return x.strip().replace("."," ")

In [ ]:
df["Symbol"] = df["Symbol"].apply(normalize)
print(set(df["Symbol"].tolist())-set(tickers))

Creation of the `.csv` file where status/rename/industry/exchange/delist info live

First create a DataFrame and export to csv. Columns:

`conID | ticker | secIdType | secId | sec_type | primary_exchange | currency | gics_sector | status | delist_date | former_ticker | last_fetch`

In [ ]:
import nest_asyncio; nest_asyncio.apply()
from src.execution.ib_connection import IBConnection
from ib_async import Stock
conn = IBConnection()
conn.connect()

print(f"Connected: {conn.is_connected()}")
print(f"Accounts: {conn.ib.managedAccounts()}")

cds = conn.ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

conn.disconnect()

In [ ]:
import nest_asyncio; nest_asyncio.apply()      # needed for ib_async in Jupyter
from ib_async import Stock
from src.execution.ib_connection import IBConnection

with IBConnection(client_id=1) as ib:           # host/port from .env
    cds = ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
    print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

## Chat phase 6a. universe
The new strategy is to build on iShares IWB daily holdings. The ETF replicates the Russell1000 and therefore provides all its constituents.

In [1]:
import pandas as pd

path = '/Users/hugo/quant-research/config/IWB_holdings.csv'
raw = pd.read_csv(path,skiprows=9,thousands=',',na_values=['-'])
raw.head()

,Ticker,Name,Sector,Asset Class,Market Value,Weight (%),Notional Value,Quantity,Price,Location,Exchange,Currency,FX Rate,Market Currency,Accrual Date
0,NVDA,NVIDIA,Information Technology,Equity,3.444626e+09,7.09,3.444626e+09,16086609.0,214.13,United States,NASDAQ,USD,1.0,USD,NaN
1,AAPL,APPLE,Information Technology,Equity,3.424375e+09,7.05,3.424375e+09,9970228.0,343.46,United States,NASDAQ,USD,1.0,USD,NaN
2,MSFT,MICROSOFT,Information Technology,Equity,2.710482e+09,5.58,2.710482e+09,5123301.0,529.05,United States,NASDAQ,USD,1.0,USD,NaN
3,AMZN,AMAZON.COM INC,Consumer Discretionary,Equity,1.672395e+09,3.44,1.672395e+09,6689581.0,250.00,United States,NASDAQ,USD,1.0,USD,NaN
4,GOOGL,ALPHABET CLASS A,Communication,Equity,1.368546e+09,2.82,1.368546e+09,4018162.0,340.59,United States,NASDAQ,USD,1.0,USD,NaN


In [2]:
# Inspect data
raw.shape, raw.dtypes

((1028, 15),
 Ticker                 str
 Name                   str
 Sector                 str
 Asset Class            str
 Market Value       float64
 Weight (%)         float64
 Notional Value     float64
 Quantity           float64
 Price              float64
 Location               str
 Exchange               str
 Currency               str
 FX Rate            float64
 Market Currency        str
 Accrual Date           str
 dtype: object)

In [3]:
# Parse date of holdings

from datetime import date, datetime
from itertools import islice
from pathlib import Path

AS_OF_LABEL = "Fund Holdings as of"
N_HEADER_LINES = 9


def parse_as_of_date(path: str | Path) -> date:
    """Return the holdings 'as of' date from the iShares CSV metadata block.

    Raises ValueError if the label is missing or the date can't be parsed.
    """
    with open(path, encoding="utf-8") as f:
        for line in islice(f, N_HEADER_LINES):
            if line.startswith(AS_OF_LABEL):
                raw = line.split(",", maxsplit=1)[1].strip().strip('"')
                return datetime.strptime(raw, "%b %d, %Y").date()
    raise ValueError(f"'{AS_OF_LABEL}' not found in first {N_HEADER_LINES} lines of {path}")

In [4]:
parse_as_of_date(path)

datetime.date(2026, 10, 16)

In [ ]:
# to check that it doesn't work and prints an understandable error
path2 = '/Users/hugo/quant-research/config/IWB_holdings_3.csv'
parse_as_of_date(path2)

In [5]:
float_vars = raw.dtypes[raw.dtypes == 'float64'].index.tolist()

In [6]:
(raw[float_vars].astype(str) == '-').any()

Market Value      False
Weight (%)        False
Notional Value    False
Quantity          False
Price             False
FX Rate           False
dtype: bool

In [7]:
raw.loc[raw['Ticker'] == 'NVDA', 'Weight (%)']

0    7.09
Name: Weight (%), dtype: float64

In [8]:
print(raw['Weight (%)'].sum())

100.09


It sums to more than 100% because the fund contains also cash and a couple of other assets

In [9]:
# Step 4
holdings = raw[raw['Asset Class'] == 'Equity']
print(holdings.shape)

(1023, 15)


In [10]:
raw[raw['Ticker'].isin(['HOLX','UWMC-R-W'])]

,Ticker,Name,Sector,Asset Class,Market Value,Weight (%),Notional Value,Quantity,Price,Location,Exchange,Currency,FX Rate,Market Currency,Accrual Date
1024,HOLX,HOLOGIC INC,Health Care,Equity,1574.05,0.0,1574.05,157405.0,0.01,United States,NO MARKET (E.G. UNLISTED),USD,1.0,USD,NaN
1025,UWMC-R-W,UWM HOLDINGS RIGHTS WHEN ISSUED EX,Financials,Equity,254.46,0.0,254.46,121173.0,0.00,United States,NYSE,USD,1.0,USD,NaN


In [11]:
exchanges = holdings["Exchange"].unique().tolist()
print(exchanges)

['NASDAQ', 'NYSE', 'Cboe BZX', 'Nyse Mkt Llc', 'NO MARKET (E.G. UNLISTED)']


In [14]:
exclude_noexchange = exchanges[4]

'NO MARKET (E.G. UNLISTED)'

In [ ]:
# creating an allowing list for tickers: uppercase letters plus a space and a class letter
# Using regex
holdings = holdings[holdings['Ticker'].str.fullmatch(r'[A-Z]+(.[A-Z])?')].copy()
print(holdings.shape)

In [ ]:
# above takes care only of non-usual tickers name. need to take care of those tickers whose exchanges are not the traditional exchanges, should be automatic


In [22]:
holdings = raw[raw['Asset Class'] == 'Equity']
holdings = holdings[holdings['Ticker'].str.fullmatch(r'[A-Z]+(.[A-Z])?')].copy()
exchanges = holdings["Exchange"].unique().tolist()
exclude_exchange = [exchanges[4]]
holdings = holdings.loc[~holdings['Exchange'].isin(exclude_exchange)].copy()

In [38]:
# create dictionary to rename and select columns
var_select = {'Ticker':'ticker', 'Name':'name', 'Sector':'sector','Market Value':'market_value','Price':'price','Exchange':'exchange'}
holdings = holdings.loc[:,var_select.keys()].rename(columns=var_select).copy()
holdings.head()

,ticker,name,sector,market_value,price,exchange
0,NVDA,NVIDIA,Information Technology,3.444626e+09,214.13,NASDAQ
1,AAPL,APPLE,Information Technology,3.424375e+09,343.46,NASDAQ
2,MSFT,MICROSOFT,Information Technology,2.710482e+09,529.05,NASDAQ
3,AMZN,AMAZON.COM INC,Consumer Discretionary,1.672395e+09,250.00,NASDAQ
4,GOOGL,ALPHABET CLASS A,Communication,1.368546e+09,340.59,NASDAQ


In [40]:
# check that tickers are unique in the final data
print(holdings.ticker.is_unique)

True


In [41]:
# solution
TICKER_PATTERN = r"[A-Z]+( [A-Z])?"          # common share, optional class letter: BRK B
LISTED_EXCHANGES = {"NYSE", "NASDAQ", "Cboe BZX", "Nyse Mkt Llc"}
COLUMNS = {"Ticker": "ticker", "Name": "name", "Sector": "sector",
           "Market Value": "market_value", "Price": "price",
           "Exchange": "exchange", "Location": "location"}

equities = raw[raw["Asset Class"] == "Equity"]

valid_ticker = equities["Ticker"].str.fullmatch(TICKER_PATTERN)
listed = equities["Exchange"].isin(LISTED_EXCHANGES)
keep = valid_ticker & listed

dropped = equities.loc[~keep, ["Ticker", "Name", "Exchange"]]
print(f"Dropped {len(dropped)} equity rows:\n{dropped.to_string(index=False)}")

holdings = equities.loc[keep, list(COLUMNS)].rename(columns=COLUMNS)
print(holdings.shape, holdings.ticker.is_unique)

Dropped 2 equity rows:
  Ticker                               Name                  Exchange
    HOLX                        HOLOGIC INC NO MARKET (E.G. UNLISTED)
UWMC-R-W UWM HOLDINGS RIGHTS WHEN ISSUED EX                      NYSE
(1021, 7) True


In [ ]:
# Step 5
from datetime import timedelta

MAX_ROWS = 1040
MIN_ROWS = 990
SECTORS = raw.Sector.unique().tolist()
SECTORS.remove('Cash and/or Derivatives')
MAX_AGE_DAYS = 7

def validate_holdings(holdings: pd.DataFrame, as_of: date, today: date):
    """
    Validates the holdings
    """
    failures = []
    if holdings.shape[0]<MIN_ROWS or holdings.shape[0]>MAX_ROWS:
        failures.append(f"Truncated download: {holdings.shape[0]} rows in DataFrame.")

    if not holdings.ticker.is_unique:
        duplicated_tickers = holdings.loc[holdings['tickers'].duplicated(),'ticker'].unique().tolist()
        failures.append(f"Duplicated tickers: {duplicated_tickers}")

    if holdings[["ticker","name","sector","price","market_value"]].isna().any().any():
        failures.append(f"Missing values recorded")

    if not ((holdings.price>0).all() and (holdings.market_value>0).all()):
        failures.append(f"Null or negative price(s)/market value(s) recorded")

    if holdings.sector.unique().tolist() not in SECTORS:
        failures.append(f"Sector not recognized")

    if as_of>today or (as_of<today-timedelta(days=7)):
        failures.append(f"Today or tomorrow not recognized")


    raise ValueError(failure for failure in failures)

In [49]:
(holdings.price>0).all()

np.True_

In [50]:
raw.Sector.unique()

<ArrowStringArray>
[ 'Information Technology',  'Consumer Discretionary',
           'Communication',             'Health Care',
              'Financials',                  'Energy',
        'Consumer Staples',             'Industrials',
               'Materials',             'Real Estate',
               'Utilities', 'Cash and/or Derivatives']
Length: 12, dtype: str

In [ ]:
# Solution step 5
from datetime import date

MIN_ROWS, MAX_ROWS = 990, 1040
MAX_AGE_DAYS = 7
REQUIRED_COLUMNS = ["ticker", "name", "sector", "price", "market_value"]
GICS_SECTORS = {
    "Communication", "Consumer Discretionary", "Consumer Staples", "Energy",
    "Financials", "Health Care", "Industrials", "Information Technology",
    "Materials", "Real Estate", "Utilities",
}


def validate_holdings(holdings: pd.DataFrame, as_of: date, today: date) -> None:
    """Check the cleaned IWB holdings are plausible before they become a snapshot.

    Raises ValueError listing every failed check.
    """
    failures = []

    n = len(holdings)
    if not MIN_ROWS <= n <= MAX_ROWS:
        failures.append(f"row count {n} outside [{MIN_ROWS}, {MAX_ROWS}]")

    dupes = holdings.loc[holdings["ticker"].duplicated(), "ticker"].unique().tolist()
    if dupes:
        failures.append(f"duplicate tickers: {dupes}")

    missing = holdings[REQUIRED_COLUMNS].isna().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        failures.append(f"missing values: {missing.to_dict()}")

    bad = holdings.loc[(holdings["price"] <= 0) | (holdings["market_value"] <= 0), "ticker"].tolist()
    if bad:
        failures.append(f"non-positive price or market value: {bad}")

    unknown = set(holdings["sector"].dropna()) - GICS_SECTORS
    if unknown:
        failures.append(f"unknown sectors: {sorted(unknown)}")

    age = (today - as_of).days
    if not 0 <= age <= MAX_AGE_DAYS:
        failures.append(f"as_of {as_of} is {age} days before today {today} (allowed 0-{MAX_AGE_DAYS})")

    if failures:
        raise ValueError("Holdings validation failed:\n- " + "\n- ".join(failures))

In [54]:
# Step 6
import requests

IWB_HOLDINGS_URL = (
    "https://www.blackrock.com/varnish-api/blk-one01-product-data/product-data/api/v1/"
    "get-fund-document?appType=PRODUCT_PAGE&appSubType=ISHARES&targetSite=us-ishares"
    "&locale=en_US&portfolioId=239707&userType=individual&component=holdings"
)

response = requests.get(IWB_HOLDINGS_URL, timeout=30)

In [57]:
print(response.status_code) # 200 means success

200


In [62]:
print(repr(response.text[:500]))

'iShares Russell 1000 ETF\nFund Holdings as of,"Nov 13, 2026"\nInception Date,"May 15, 2000"\nShares Outstanding,"116,250,000.00"\nStock,"-"\nBond,"-"\nCash,"-"\nOther,"-"\n\nTicker,Name,Type,Sector,Asset Class,Market Value,Notional Value,Quantity,Price,Location,Exchange,Currency,FX Rate,Market Currency,Accrual Date,Market Weight,Notional Weight\n"AAPL","APPLE","EQUITY","Information Technology","Equity","3,548,169,884.32","3,548,169,884.32","9,959,216.00","356.27","United States","NASDAQ","USD","1.00","USD'


In [66]:
header_pos = response.text.find("Ticker")
n_lines = response.text[:header_pos].count("\n")

165

In [59]:
print(response.content[:100])

b'iShares Russell 1000 ETF\nFund Holdings as of,"Nov 13, 2026"\nInception Date,"May 15, 2000"\nShares Out'


In [ ]:
from pathlib import Path
raw_bytes_path = Path("data/production/universe/raw")
raw_bytes_path.mkdir(parents=True, exist_ok=True)
file_path = raw_bytes_path / f"IWB_holdings_{as_of}.csv"
file_path.write_bytes(response.content)